### Wandertag – Lösung Schritt für Schritt

*43. Bundeswettbewerb Informatik (2024), 1. Runde, Aufgabe 3*

Einmal im Jahr veranstaltet die Informatik-Gesundheitskasse (IGK) für ihre Mitglieder einen Wandertag. Leider hat sich in den letzten Jahren
gezeigt, dass viele Mitglieder nicht teilnehmen wollten, weil die Strecke zu lang oder zu kurz war.

Deshalb hat die Krankenkasse jetzt erstmalig eine Befragung zu dem Wandertag durchgeführt. Jedes Mitglied hat dabei eine minimale und maximale
Streckenlänge angegeben: Liegt die Länge der angebotenen Strecke in diesem Bereich, nimmt das Mitglied teil.

Es wird schnell offensichtlich, dass viel mehr Mitglieder teilnehmen, wenn es mehrere unterschiedlich lange Strecken zur Auswahl gibt.

#### Aufgabe 3

Unterstütze die IGK bei der Planung und schreibe ein Programm, das drei Streckenlängen so berechnet, dass möglichst viele Mitglieder
teilnehmen. Es soll ausgeben, wie viele Mitglieder insgesamt teilnehmen würden und bei welcher Streckenlänge welche Mitglieder teilnehmen könnten.

Wende dein Programm mindestens auf alle Beispiele an, die du auf den BWINF-Webseiten findest, und dokumentiere die Ergebnisse.

#### Beispieldaten

Im Ordner `beispieldaten` stehen `wandern1.txt` bis `wandern7.txt`. Die erste Zeile enthält die Zahl `n` der Personen. In den folgenden `n` Zeilen
stehen die kürzeste und die längste gewünschte Strecke einer Person in Metern. Beide Grenzen gehören dazu.

```
3
500 1000
100 200
100 600
```

Person 1 möchte mindestens 500 m und höchstens 1000 m laufen und käme auch bei genau 500 m oder 1000 m mit. Person 2 wünscht sich 100 m bis 200 m, Person 3
100 m bis 600 m. Für `wandern6.txt` (500 Personen) und `wandern7.txt` (800 Personen) braucht ein einfaches Programm viel Zeit.

## Was dir bei dieser Aufgabe hilft

**Algorithmische Kenntnisse**
- **Intervalle:** Jeder Wunsch ist ein Bereich `[von, bis]`. Eine Strecke der Länge `x` gefällt der Person, wenn `von ≤ x ≤ bis` ist. Wir suchen drei Längen, sodass
  möglichst viele Intervalle mindestens eine davon enthalten.
- **Kandidaten verkleinern:** Von den vielen möglichen Streckenlängen reichen **die unteren Grenzen der Wünsche**. Das macht aus bis zu 100 000 möglichen Längen höchstens `n`.
- **Brute Force:** Alle Kombinationen von drei Kandidaten durchprobieren. Das ist einfach, aber bei `n` Kandidaten sind es etwa `n³ / 6` Kombinationen.
- **Dynamische Programmierung (DP):** Das Problem in kleinere zerlegen und Teilergebnisse speichern: Was ist das beste Ergebnis mit einer, mit zwei, mit drei Strecken, wenn die längste
  Strecke ein bestimmter Kandidat ist?
- **Tabelle der Schnittmengen:** Wie viele Wünsche decken **zwei** Kandidaten gemeinsam ab? Mit dieser Tabelle zählt man, wie viele Wünsche eine neue Strecke **zusätzlich** abdeckt, ohne
  alle Wünsche anzuschauen.

**Nützliche Funktionen und Bibliotheken**
- Listen, Tupel `(von, bis)`, `sorted`, `set`
- Verschachtelte Schleifen und zweidimensionale Listen (Listen von Listen)

## Die Idee

1. Die Wünsche einlesen und eine Funktion bauen, die zählt, wie viele Wünsche von einer Menge von Strecken abgedeckt werden.
2. Die **Kandidaten** für die Strecken auf die unteren Grenzen der Wünsche beschränken.
3. **Brute Force:** alle Dreiergruppen von Kandidaten ausprobieren (nur für kleine Beispiele).
4. **DP:** Für `i = 1, 2, 3` Strecken und jeden Kandidaten als längste Strecke das beste Ergebnis berechnen, mit der Schnittmengentabelle schnell.
5. Die drei Strecken und die Teilnehmer ausgeben und mit der Zahl vergleichen, die Brute Force liefert.

## Schritt 1: Einlesen und Zählen

Jeder Wunsch ist ein Tupel `(von, bis)`. `abgedeckt(laenge, wunsch)` sagt, ob die Streckenlänge im Wunsch liegt (beide Grenzen gehören dazu).
`teilnehmer(laengen, wuensche)` zählt die Wünsche, die von **mindestens einer** der Längen abgedeckt werden. Ein Wunsch zählt nur einmal, auch wenn zwei Strecken in seinen Bereich fallen.

In [ ]:
def lies_wuensche(dateiname):
    with open(dateiname, encoding='utf-8') as f:
        zahlen = []
        for text in f.read().split():
            zahlen.append(int(text))
    wuensche = []
    for i in range(zahlen[0]):
        wuensche.append((zahlen[1 + 2 * i], zahlen[2 + 2 * i]))
    return wuensche


def abgedeckt(laenge, wunsch):
    return wunsch[0] <= laenge <= wunsch[1]


def teilnehmer(laengen, wuensche):
    anzahl = 0
    for wunsch in wuensche:
        for laenge in laengen:
            if abgedeckt(laenge, wunsch):
                anzahl = anzahl + 1
                break                                      # jeder Wunsch zählt nur einmal
    return anzahl


wuensche = lies_wuensche('beispieldaten/wandern1.txt')
print(wuensche)
print('Bei 22, 51 und 64 Metern kommen', teilnehmer([22, 51, 64], wuensche), 'von', len(wuensche), 'Personen mit.')

[(12, 35), (22, 45), (46, 46), (48, 62), (51, 57), (64, 64), (64, 71)]
Bei 22, 51 und 64 Metern kommen 6 von 7 Personen mit.


## Schritt 2: Welche Streckenlängen kommen in Frage?

Streckenlängen gibt es sehr viele (bis zu 100 000 Meter), aber wir müssen nicht alle ausprobieren. Betrachte eine Strecke `p`. Verschieben wir sie **nach links bis zur nächsten unteren Grenze** `L` eines
Wunsches (die größte untere Grenze, die höchstens `p` ist), dann bleiben alle Wünsche, die `p` abdeckte, abgedeckt. Denn bei jedem von ihnen ist die untere Grenze höchstens `L`
und die obere Grenze mindestens `p ≥ L`. Es genügt also, **nur die unteren Grenzen der Wünsche** als Kandidaten zu betrachten (jede nur einmal).

In [ ]:
def kandidaten_von(wuensche):
    untere = set()
    for wunsch in wuensche:
        untere.add(wunsch[0])
    return sorted(untere)


kandidaten = kandidaten_von(wuensche)
print('Kandidaten:', kandidaten)
for nr in range(1, 8):
    w = lies_wuensche('beispieldaten/wandern' + str(nr) + '.txt')
    k = kandidaten_von(w)
    print('wandern' + str(nr) + '.txt:', len(w), 'Personen,', len(k), 'Kandidaten, kleinste und größte Grenze:', min(min(x) for x in w), max(max(x) for x in w))

Kandidaten: [12, 22, 46, 48, 51, 64]
wandern1.txt: 7 Personen, 6 Kandidaten, kleinste und größte Grenze: 12 71
wandern2.txt: 6 Personen, 4 Kandidaten, kleinste und größte Grenze: 10 90
wandern3.txt: 10 Personen, 10 Kandidaten, kleinste und größte Grenze: 16 96
wandern4.txt: 100 Personen, 93 Kandidaten, kleinste und größte Grenze: 1 998
wandern5.txt: 200 Personen, 200 Kandidaten, kleinste und größte Grenze: 63 99992
wandern6.txt: 500 Personen, 497 Kandidaten, kleinste und größte Grenze: 76 99966
wandern7.txt: 800 Personen, 799 Kandidaten, kleinste und größte Grenze: 29 99998


## Schritt 3: Brute Force für kleine Beispiele

Mit den Kandidaten ist die einfachste Lösung: **alle Dreiergruppen** `K[i], K[j], K[k]` mit `i ≤ j ≤ k` ausprobieren und die beste merken. `i ≤ j ≤ k` erlaubt auch, dass eine Länge
mehrmals vorkommt, falls es weniger als drei verschiedene Kandidaten gibt. Für kleine Beispiele geht das gut.

In [ ]:
def brute_force(wuensche):
    kandidaten = kandidaten_von(wuensche)
    bester = -1
    beste_laengen = None
    for i in range(len(kandidaten)):
        for j in range(i, len(kandidaten)):
            for k in range(j, len(kandidaten)):
                laengen = [kandidaten[i], kandidaten[j], kandidaten[k]]
                anzahl = teilnehmer(laengen, wuensche)
                if anzahl > bester:
                    bester = anzahl
                    beste_laengen = laengen
    return bester, beste_laengen


for nr in range(1, 4):
    wuensche = lies_wuensche('beispieldaten/wandern' + str(nr) + '.txt')
    bester, laengen = brute_force(wuensche)
    print('wandern' + str(nr) + '.txt:', bester, 'von', len(wuensche), 'Personen, Längen', laengen)

wandern1.txt: 6 von 7 Personen, Längen [22, 51, 64]
wandern2.txt: 6 von 6 Personen, Längen [10, 60, 90]
wandern3.txt: 10 von 10 Personen, Längen [19, 66, 92]


Bei `wandern1.txt` und `wandern2.txt` können mit drei Strecken alle bis auf eine Person (6 von 7) beziehungsweise alle (6 von 6) mitkommen. Bei `wandern3.txt` kommen alle 10 mit.

Bei `wandern7.txt` mit 800 Personen gibt es 800 Kandidaten und damit etwa 85 Millionen Dreiergruppen, und für jede müsste man alle 800 Wünsche prüfen. Das dauert viel zu lange. Wir brauchen
etwas Besseres.

## Schritt 4: Die Schnittmengentabelle

Die Kandidaten seien aufsteigend sortiert: `K[0] < K[1] < …`. Wir bauen eine Tabelle `S`, bei der `S[k][j]` für `k ≤ j` die Zahl der Wünsche ist, die **sowohl `K[k]` als auch `K[j]` abdecken**. Auf der
Diagonale steht dann `S[j][j]`: die Zahl der Wünsche, die `K[j]` allein abdeckt.

Ein Wunsch `[von, bis]` deckt beide ab, wenn `von ≤ K[k]` und `bis ≥ K[j]` gilt. Wir wandern durch die Kandidaten von links nach rechts. Zu jedem `k` nehmen wir die Wünsche mit `von ≤ K[k]` in eine
Zählliste `zaehler` auf. `zaehler[t]` ist die Zahl der aufgenommenen Wünsche, bei denen **genau `t` Kandidaten höchstens so groß sind wie `bis`**. Ein Wunsch deckt `K[j]` genau dann ab,
wenn dieser Wert mindestens `j + 1` ist. `S[k][j]` ist also die Summe von `zaehler[t]` für alle `t ≥ j + 1`. Das berechnen wir für jedes `j` von rechts nach links in einem Durchlauf.

In [ ]:
def schnitt_tabelle(kandidaten, wuensche):
    m = len(kandidaten)
    sortiert = sorted(wuensche)                            # nach der unteren Grenze
    # t_von_bis: wie viele Kandidaten sind höchstens so groß wie die obere Grenze des Wunsches?
    anzahl_kleiner = {}
    for wunsch in wuensche:
        t = 0
        for kandidat in kandidaten:
            if kandidat <= wunsch[1]:
                t = t + 1
        anzahl_kleiner[wunsch] = t
    zaehler = [0] * (m + 1)
    naechster = 0                                          # der nächste Wunsch in der sortierten Liste
    S = []
    for k in range(m):
        while naechster < len(sortiert) and sortiert[naechster][0] <= kandidaten[k]:
            zaehler[anzahl_kleiner[sortiert[naechster]]] = zaehler[anzahl_kleiner[sortiert[naechster]]] + 1
            naechster = naechster + 1
        zeile = [0] * m
        summe = 0
        for j in range(m - 1, k - 1, -1):                  # von rechts nach links: Summe der zaehler[t] für t >= j + 1
            summe = summe + zaehler[j + 1]
            zeile[j] = summe
        S.append(zeile)
    return S


kandidaten = kandidaten_von(lies_wuensche('beispieldaten/wandern1.txt'))
wuensche = lies_wuensche('beispieldaten/wandern1.txt')
S = schnitt_tabelle(kandidaten, wuensche)
print('Kandidaten:', kandidaten)
for k in range(len(kandidaten)):
    print(kandidaten[k], S[k])

Kandidaten: [12, 22, 46, 48, 51, 64]
12 [1, 1, 0, 0, 0, 0]
22 [0, 2, 0, 0, 0, 0]
46 [0, 0, 1, 0, 0, 0]
48 [0, 0, 0, 1, 1, 0]
51 [0, 0, 0, 0, 2, 0]
64 [0, 0, 0, 0, 0, 2]


Die Diagonale `S[j][j]` ist die Zahl der Wünsche, die ein Kandidat allein abdeckt. Zum Beispiel deckt der Kandidat 22 genau 2 Wünsche ab (`[12, 35]` und `[22, 45]`).

## Schritt 5: Dynamische Programmierung

`D[i][j]` sei die **größte Zahl abgedeckter Wünsche**, wenn wir `i` Kandidaten wählen und `K[j]` der **größte** der gewählten ist.

- **Eine Strecke (`i = 1`):** `D[1][j] = S[j][j]`.
- **Mehr Strecken:** Der Vorgänger ist ein kleinerer Kandidat `K[k]` mit `k < j`. Die Strecke `K[j]` deckt `S[j][j]` Wünsche ab, davon sind `S[k][j]` schon von `K[k]` abgedeckt. Neu sind also
  `S[j][j] − S[k][j]`. Der Rest ist auch von den noch früheren Strecken schon abgedeckt, denn ein Wunsch, der einen früheren Kandidaten und `K[j]` abdeckt, deckt auch alles dazwischen
  ab, also auch `K[k]`.

  `D[i][j] = größtes D[i − 1][k] + S[j][j] − S[k][j]` über alle `k < j`.

Wir merken uns auch das beste `k` (in `vorgaenger`), um die Strecken am Ende zu finden. Das Ergebnis ist das größte `D[3][j]`, und mit `vorgaenger` gehen wir von `j` aus zurück. Mit
weniger als drei Kandidaten wählen wir einfach alle.

In [ ]:
def waehle_strecken(wuensche, anzahl_strecken=3):
    kandidaten = kandidaten_von(wuensche)
    m = len(kandidaten)
    if m <= anzahl_strecken:
        return teilnehmer(kandidaten, wuensche), kandidaten
    S = schnitt_tabelle(kandidaten, wuensche)
    D = [[-1] * m]                                         # Zeile 0 wird nicht gebraucht
    vorgaenger = [[-1] * m]
    erste = []
    for j in range(m):
        erste.append(S[j][j])
    D.append(erste)                                        # eine Strecke
    vorgaenger.append([-1] * m)
    for i in range(2, anzahl_strecken + 1):
        zeile = [-1] * m
        woher = [-1] * m
        for j in range(m):
            for k in range(j):
                if D[i - 1][k] >= 0:
                    wert = D[i - 1][k] + S[j][j] - S[k][j]
                    if wert > zeile[j]:
                        zeile[j] = wert
                        woher[j] = k
        D.append(zeile)
        vorgaenger.append(woher)
    bestes_j = 0
    for j in range(m):
        if D[anzahl_strecken][j] > D[anzahl_strecken][bestes_j]:
            bestes_j = j
    laengen = []
    j = bestes_j
    for i in range(anzahl_strecken, 0, -1):
        laengen.append(kandidaten[j])
        j = vorgaenger[i][j]
    laengen.reverse()
    return D[anzahl_strecken][bestes_j], laengen


for nr in range(1, 4):
    wuensche = lies_wuensche('beispieldaten/wandern' + str(nr) + '.txt')
    anzahl, laengen = waehle_strecken(wuensche)
    print('wandern' + str(nr) + '.txt: DP', anzahl, 'Längen', laengen, ', gezählt:', teilnehmer(laengen, wuensche), ', Brute Force:', brute_force(wuensche)[0])

wandern1.txt: DP 6 Längen [22, 51, 64] , gezählt: 6 , Brute Force: 6
wandern2.txt: DP 6 Längen [10, 60, 90] , gezählt: 6 , Brute Force: 6
wandern3.txt: DP 10 Längen [19, 66, 92] , gezählt: 10 , Brute Force: 10


Die DP findet bei den kleinen Beispielen dasselbe wie Brute Force, und die Zahl stimmt mit dem Zählen der Teilnehmer für die gefundenen Längen überein.

**Aufwand:** Die Tabelle `S` hat `|K|²` Einträge, und das DP rechnet für jedes `i` und jedes Paar `k < j` genau eine Zahl aus, also etwa `3 · |K|² / 2` Schritte. Bei 800 Kandidaten sind das
knapp 1 Million Schritte statt 85 Millionen Dreiergruppen mal 800 Wünsche.

## Schritt 6: Wer kommt zu welcher Strecke?

Die Aufgabe will auch wissen, **bei welcher Streckenlänge welche Mitglieder** teilnehmen. Ein Mitglied kann zu mehreren Strecken passen. Wir ordnen jedes Mitglied der **ersten** Strecke zu, die zu ihm passt, damit jedes
nur einmal vorkommt und die Summe die Teilnehmerzahl ist. Mitglieder zählen wir ab 0 in der Reihenfolge der Datei.

In [ ]:
def verteile(laengen, wuensche):
    gruppen = []
    for laenge in laengen:
        gruppen.append([])
    for nr in range(len(wuensche)):
        for g in range(len(laengen)):
            if abgedeckt(laengen[g], wuensche[nr]):
                gruppen[g].append(nr)
                break                                      # nur die erste passende Strecke
    return gruppen


wuensche = lies_wuensche('beispieldaten/wandern1.txt')
anzahl, laengen = waehle_strecken(wuensche)
gruppen = verteile(laengen, wuensche)
print('Es nehmen', anzahl, 'von', len(wuensche), 'Mitgliedern teil.')
for g in range(len(laengen)):
    print('Strecke', laengen[g], 'm:', len(gruppen[g]), 'Mitglieder', gruppen[g])

Es nehmen 6 von 7 Mitgliedern teil.
Strecke 22 m: 2 Mitglieder [0, 1]
Strecke 51 m: 2 Mitglieder [3, 4]
Strecke 64 m: 2 Mitglieder [5, 6]


## Schritt 7: Alle Beispiele

Für jedes Beispiel geben wir die Teilnehmerzahl und die drei Streckenlängen aus. Bei den größeren zeigen wir nur die Zahl der Mitglieder je Strecke. Zur Kontrolle zählen wir für die
gefundenen Längen mit `teilnehmer` nach.

In [ ]:
ergebnisse = {}
for nr in range(1, 8):
    wuensche = lies_wuensche('beispieldaten/wandern' + str(nr) + '.txt')
    anzahl, laengen = waehle_strecken(wuensche)
    gruppen = verteile(laengen, wuensche)
    ergebnisse[nr] = (wuensche, anzahl, laengen, gruppen)
    je_strecke = []
    for g in gruppen:
        je_strecke.append(len(g))
    print('wandern' + str(nr) + '.txt:', anzahl, 'von', len(wuensche), 'Mitgliedern, Strecken', laengen, 'm, je Strecke', je_strecke,
          ', nachgezählt:', teilnehmer(laengen, wuensche))

wandern1.txt: 6 von 7 Mitgliedern, Strecken [22, 51, 64] m, je Strecke [2, 2, 2] , nachgezählt: 6
wandern2.txt: 6 von 6 Mitgliedern, Strecken [10, 60, 90] m, je Strecke [2, 3, 1] , nachgezählt: 6
wandern3.txt: 10 von 10 Mitgliedern, Strecken [19, 66, 92] m, je Strecke [3, 4, 3] , nachgezählt: 10
wandern4.txt: 79 von 100 Mitgliedern, Strecken [524, 811, 922] m, je Strecke [38, 28, 13] , nachgezählt: 79
wandern5.txt: 153 von 200 Mitgliedern, Strecken [36696, 60828, 88584] m, je Strecke [86, 38, 29] , nachgezählt: 153
wandern6.txt: 330 von 500 Mitgliedern, Strecken [42834, 74810, 92920] m, je Strecke [168, 103, 59] , nachgezählt: 330
wandern7.txt: 551 von 800 Mitgliedern, Strecken [39520, 76088, 91584] m, je Strecke [285, 177, 89] , nachgezählt: 551


**Ergebnis:** Die Teilnehmerzahlen stimmen mit den offiziellen Lösungshinweisen überein:

| Beispiel | Teilnehmer |
|---|---|
| wandern1 | 6 von 7 |
| wandern2 | 6 von 6 |
| wandern3 | 10 von 10 |
| wandern4 | 79 von 100 |
| wandern5 | 153 von 200 |
| wandern6 | 330 von 500 |
| wandern7 | 551 von 800 |

Auch die Streckenlängen und die Mitglieder je Strecke stimmen bei allen sieben Beispielen mit den Lösungshinweisen überein (zum Beispiel 524, 811 und 922 m bei `wandern4.txt`), weil wir wie dort die
unteren Grenzen als Kandidaten nehmen. Im Allgemeinen können sie sich unterscheiden: Oft decken mehrere Längen genau dieselben Wünsche ab, und die Zuordnung der Mitglieder zu den Strecken ist nur eine von mehreren
möglichen.

Die vollständigen Listen der Mitglieder je Strecke für die kleinen Beispiele:

In [ ]:
for nr in range(1, 5):
    wuensche, anzahl, laengen, gruppen = ergebnisse[nr]
    print('wandern' + str(nr) + '.txt: insgesamt', anzahl, 'von', len(wuensche))
    for g in range(len(laengen)):
        text = ' '.join(str(x) for x in gruppen[g])
        print('  Strecke', str(laengen[g]).rjust(5), 'm,', str(len(gruppen[g])).rjust(2), 'Mitglieder:', text)

wandern1.txt: insgesamt 6 von 7
  Strecke    22 m,  2 Mitglieder: 0 1
  Strecke    51 m,  2 Mitglieder: 3 4
  Strecke    64 m,  2 Mitglieder: 5 6
wandern2.txt: insgesamt 6 von 6
  Strecke    10 m,  2 Mitglieder: 4 5
  Strecke    60 m,  3 Mitglieder: 0 2 3
  Strecke    90 m,  1 Mitglieder: 1
wandern3.txt: insgesamt 10 von 10
  Strecke    19 m,  3 Mitglieder: 6 8 9
  Strecke    66 m,  4 Mitglieder: 0 1 2 5
  Strecke    92 m,  3 Mitglieder: 3 4 7
wandern4.txt: insgesamt 79 von 100
  Strecke   524 m, 38 Mitglieder: 3 6 7 9 10 11 12 19 20 23 24 25 34 37 38 39 40 42 47 54 56 57 58 60 64 65 67 71 78 80 81 82 83 90 91 97 98 99
  Strecke   811 m, 28 Mitglieder: 0 1 4 5 8 13 14 16 18 21 26 27 29 31 32 43 48 51 53 55 59 62 73 74 75 87 88 96
  Strecke   922 m, 13 Mitglieder: 22 30 35 36 46 50 66 69 72 86 89 92 94


## Für Neugierige

**Warum reichen die unteren Grenzen?** Das Argument aus Schritt 2 zeigt, dass man jede Strecke nach links schieben darf, bis sie auf einer unteren Grenze liegt. Man könnte genauso gut die oberen Grenzen
als Kandidaten nehmen und nach rechts schieben.

**Laufzeit:** Brute Force braucht etwa `|K|³ · n / 6` Schritte, unsere DP mit Schnittmengentabelle ungefähr `|K|² + n · |K|` (die Tabelle selbst wird bei uns noch mit einer einfachen Zählung aufgebaut). Die Lösungshinweise
bauen die Tabelle in `O(n²)` auf, wenn man die Wünsche sortiert durchläuft.

**Mehr als drei Strecken:** Die DP ändert sich nicht: Man berechnet einfach mehr Zeilen `D[i]`. Der Aufwand wächst nur **linear** mit der Zahl der Strecken, bei Brute Force dagegen exponentiell.
Mit `waehle_strecken(wuensche, 5)` rechnet das Programm fünf Strecken.

**Schneller mit einer anderen Idee:** Das Problem ist verwandt mit dem Finden von `k` Punkten, die möglichst viele Intervalle „aufspießen“ (*Interval Stabbing*). Mit einer Datenstruktur für Bereiche, einem
*Segmentbaum*, lässt sich die DP noch schneller machen. Das braucht deutlich mehr Programmierkenntnisse.

**MiniZinc:** Hier haben wir keinen Abschnitt. Mit nur drei Streckenlängen ist es ein kleines DP-Problem, für das ein Solver keinen Vorteil bringt.